# Module 0.1: API Setup and Configuration

<span class="badge blue">Agentic AI</span> <span class="badge amber">~15 min</span> <span class="badge mint">Hands-on</span>

## Learning Objectives

By completing this notebook, you will:
1. Set up API keys for multiple LLM providers (OpenAI, Anthropic)
2. Make your first API calls to different models
3. Understand request/response structures
4. Compare basic capabilities across providers
5. Implement proper error handling for API calls

## Prerequisites

- Python 3.10 or higher
- API key from at least one provider (OpenAI or Anthropic)
- Basic understanding of REST APIs

## Estimated Time: 45 minutes

---

## 1. Setup & Imports

We'll install and import the necessary libraries for interacting with LLM APIs. The key libraries are:
- `openai`: OpenAI's official Python client
- `anthropic`: Anthropic's official Python client
- `python-dotenv`: For secure API key management

In [ ]:
# Setup: Course styling and plot theme
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname("__file__"), "../../../../.."))

from resources.notebook_style import apply_course_theme
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()
print("Course theme applied.")

In [ ]:
# Install required packages
# Uncomment the line below if running for the first time
# !pip install openai anthropic python-dotenv

import os
import json
from typing import Dict, List, Optional
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

print("✓ Imports successful")

## 2. API Key Configuration

**Security Best Practice**: Never hardcode API keys in your code. Use environment variables instead.

Create a `.env` file in your project root with:
```
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
```

For this tutorial, we'll use mock keys to demonstrate the setup process.

In [ ]:
# Purpose: Configure API keys securely
# Key Concept: Environment-based configuration prevents accidental key exposure

def setup_api_keys():
    """
    Load and validate API keys from environment variables.
    
    Returns
    -------
    dict
        Dictionary containing available API keys
    """
    keys = {}
    
    # Check for OpenAI key
    openai_key = os.getenv('OPENAI_API_KEY')
    if openai_key and openai_key.startswith('sk-'):
        keys['openai'] = openai_key
        print("✓ OpenAI API key found")
    else:
        print("✗ OpenAI API key not found or invalid")
    
    # Check for Anthropic key
    anthropic_key = os.getenv('ANTHROPIC_API_KEY')
    if anthropic_key and anthropic_key.startswith('sk-ant-'):
        keys['anthropic'] = anthropic_key
        print("✓ Anthropic API key found")
    else:
        print("✗ Anthropic API key not found or invalid")
    
    if not keys:
        print("\n⚠️  No valid API keys found. Set at least one in your .env file.")
    
    return keys

api_keys = setup_api_keys()

## 3. OpenAI API Setup

OpenAI provides GPT-4 and GPT-3.5 models. The API uses a simple chat completion interface where you send messages and receive responses.

**Key Components**:
- **Model**: Specifies which LLM to use (e.g., `gpt-4`, `gpt-3.5-turbo`)
- **Messages**: List of conversation turns with roles (system, user, assistant)
- **Temperature**: Controls randomness (0 = deterministic, 1 = creative)

In [ ]:
# Purpose: Create OpenAI client and test basic completion
# Key Concept: Chat completions use a message-based interface

def test_openai_api(api_key: str) -> Dict:
    """
    Test OpenAI API with a simple completion.
    
    Parameters
    ----------
    api_key : str
        OpenAI API key
    
    Returns
    -------
    dict
        Response metadata including content and token usage
    """
    try:
        from openai import OpenAI
        
        # Initialize client
        client = OpenAI(api_key=api_key)
        
        # Create a simple chat completion
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": "Say 'Hello from OpenAI!' and nothing else."}
            ],
            temperature=0,
            max_tokens=20
        )
        
        # Extract response details
        result = {
            "content": response.choices[0].message.content,
            "model": response.model,
            "tokens_used": response.usage.total_tokens,
            "finish_reason": response.choices[0].finish_reason
        }
        
        print("✓ OpenAI API test successful")
        print(f"Response: {result['content']}")
        print(f"Tokens used: {result['tokens_used']}")
        
        return result
        
    except Exception as e:
        print(f"✗ OpenAI API test failed: {str(e)}")
        return {"error": str(e)}

# Test if OpenAI key is available
if 'openai' in api_keys:
    openai_result = test_openai_api(api_keys['openai'])
else:
    print("Skipping OpenAI test - no API key configured")
    openai_result = None

## 4. Anthropic API Setup

Anthropic provides the Claude family of models (Claude 3.5 Sonnet, Claude 3 Opus). The API is similar to OpenAI but with some differences in structure.

**Key Differences from OpenAI**:
- System prompt is a separate parameter (not in messages list)
- Response structure uses `content` blocks
- Different token counting methodology

In [ ]:
# Purpose: Create Anthropic client and test basic completion
# Key Concept: Claude API separates system prompts from message history

def test_anthropic_api(api_key: str) -> Dict:
    """
    Test Anthropic API with a simple completion.
    
    Parameters
    ----------
    api_key : str
        Anthropic API key
    
    Returns
    -------
    dict
        Response metadata including content and token usage
    """
    try:
        import anthropic
        
        # Initialize client
        client = anthropic.Anthropic(api_key=api_key)
        
        # Create a simple message
        response = client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=20,
            temperature=0,
            system="You are a helpful assistant.",
            messages=[
                {"role": "user", "content": "Say 'Hello from Claude!' and nothing else."}
            ]
        )
        
        # Extract response details
        result = {
            "content": response.content[0].text,
            "model": response.model,
            "tokens_used": response.usage.input_tokens + response.usage.output_tokens,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "stop_reason": response.stop_reason
        }
        
        print("✓ Anthropic API test successful")
        print(f"Response: {result['content']}")
        print(f"Tokens used: {result['tokens_used']} (input: {result['input_tokens']}, output: {result['output_tokens']})")
        
        return result
        
    except Exception as e:
        print(f"✗ Anthropic API test failed: {str(e)}")
        return {"error": str(e)}

# Test if Anthropic key is available
if 'anthropic' in api_keys:
    anthropic_result = test_anthropic_api(api_keys['anthropic'])
else:
    print("Skipping Anthropic test - no API key configured")
    anthropic_result = None

## 5. Error Handling Patterns

Robust API interactions require handling various failure modes:
- **Rate limits**: Too many requests in a short time
- **Invalid keys**: Authentication failures
- **Network errors**: Timeouts and connection issues
- **Model errors**: Invalid parameters or malformed requests

We'll implement a retry mechanism with exponential backoff.

In [ ]:
# Purpose: Implement robust error handling with retries
# Key Concept: Exponential backoff prevents overwhelming rate-limited APIs

import time
from typing import Callable, Any

def call_with_retry(
    api_function: Callable,
    max_retries: int = 3,
    initial_delay: float = 1.0
) -> Any:
    """
    Call an API function with exponential backoff retry logic.
    
    Parameters
    ----------
    api_function : callable
        Function to call (should take no arguments)
    max_retries : int
        Maximum number of retry attempts
    initial_delay : float
        Initial delay in seconds before first retry
    
    Returns
    -------
    any
        Result from successful API call
    
    Raises
    ------
    Exception
        If all retries are exhausted
    """
    delay = initial_delay
    
    for attempt in range(max_retries + 1):
        try:
            # Attempt the API call
            result = api_function()
            return result
            
        except Exception as e:
            error_msg = str(e).lower()
            
            # Check if error is retryable
            is_rate_limit = 'rate limit' in error_msg or '429' in error_msg
            is_server_error = '500' in error_msg or '503' in error_msg
            is_timeout = 'timeout' in error_msg
            
            if not (is_rate_limit or is_server_error or is_timeout):
                # Non-retryable error (e.g., invalid key, bad request)
                raise
            
            if attempt == max_retries:
                # All retries exhausted
                raise Exception(f"Max retries ({max_retries}) exceeded: {e}")
            
            # Wait with exponential backoff
            print(f"Attempt {attempt + 1} failed: {e}. Retrying in {delay:.1f}s...")
            time.sleep(delay)
            delay *= 2  # Exponential backoff

# Example usage
def example_api_call():
    """Simulated API call that might fail."""
    import random
    if random.random() < 0.3:  # 30% failure rate for demo
        raise Exception("Rate limit exceeded (429)")
    return {"status": "success"}

# Demonstrate retry logic
try:
    result = call_with_retry(example_api_call, max_retries=2)
    print(f"✓ API call succeeded: {result}")
except Exception as e:
    print(f"✗ API call failed after retries: {e}")

## Hands-On Exercises

Now it's time to apply what you've learned. Complete the following exercises to solidify your understanding.

### Exercise 1.1: Create a Unified API Client

**Task**: Create a function that can call either OpenAI or Anthropic based on a provider parameter.

**Expected Output**: A function that returns consistent response format regardless of provider.

**Hints**:
<details>
<summary>Hint 1</summary>
Create a dictionary to map provider names to their respective API call functions.
</details>

<details>
<summary>Hint 2</summary>
Normalize the response structure by extracting common fields (content, tokens, model).
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def unified_llm_call(provider: str, prompt: str, api_key: str) -> Dict:
    """
    Call LLM API with unified interface.
    
    Parameters
    ----------
    provider : str
        Either 'openai' or 'anthropic'
    prompt : str
        The user prompt
    api_key : str
        API key for the provider
    
    Returns
    -------
    dict
        Normalized response with keys: content, tokens, model
    """
    # TODO: Implement this function
    pass

your_answer = unified_llm_call  # Don't change this line

The test below checks that `unified_llm_call` is a callable with the correct signature: `provider`, `prompt`, and `api_key` parameters. It does not make a live API call — it verifies the function interface so learners can move forward before wiring up real credentials.

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_1_1():
    assert your_answer is not None, "Don't forget to implement your solution!"
    assert callable(your_answer), "unified_llm_call should be a function"
    
    # Test with mock implementation
    import inspect
    sig = inspect.signature(your_answer)
    params = list(sig.parameters.keys())
    
    assert len(params) >= 3, "Function should accept at least 3 parameters"
    assert 'provider' in params or params[0] == 'provider', "First parameter should be 'provider'"
    assert 'prompt' in params, "Function should have a 'prompt' parameter"
    assert 'api_key' in params, "Function should have an 'api_key' parameter"
    
    print("✓ Exercise 1.1 passed!")

test_exercise_1_1()

### Exercise 1.2: Implement Token Counter

**Task**: Create a function that estimates token count for a given text (before making an API call).

**Expected Output**: Function returns approximate token count using simple heuristic: `len(text.split()) * 1.3`

**Hints**:
<details>
<summary>Hint 1</summary>
Average English word is about 1.3 tokens for GPT models.
</details>

<details>
<summary>Hint 2</summary>
Use text.split() to count words, then multiply by 1.3 and round up.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def estimate_tokens(text: str) -> int:
    """
    Estimate token count for text.
    
    Parameters
    ----------
    text : str
        Input text
    
    Returns
    -------
    int
        Estimated token count
    """
    # TODO: Implement this function
    pass

exercise_1_2_answer = estimate_tokens  # Don't change this line

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_1_2():
    assert exercise_1_2_answer is not None, "Don't forget to implement your solution!"
    
    # Test with known examples
    test_text = "Hello world"  # 2 words -> ~2.6 tokens -> rounds to 3
    result = exercise_1_2_answer(test_text)
    
    assert isinstance(result, int), "Function should return an integer"
    assert result >= 2 and result <= 4, f"Expected ~2-4 tokens for '{test_text}', got {result}"
    
    # Test empty string
    assert exercise_1_2_answer("") == 0, "Empty string should have 0 tokens"
    
    # Test longer text
    long_text = "The quick brown fox jumps over the lazy dog"  # 9 words -> ~12 tokens
    result = exercise_1_2_answer(long_text)
    assert result >= 10 and result <= 14, f"Expected ~10-14 tokens for 9-word sentence, got {result}"
    
    print("✓ Exercise 1.2 passed!")

test_exercise_1_2()

### Exercise 1.3: Cost Calculator

**Task**: Create a function that calculates the cost of an API call given token counts and pricing.

**Expected Output**: Function returns cost in dollars (float) for given input/output tokens.

**Pricing Reference**:
- GPT-3.5-turbo: $0.50 / 1M input tokens, $1.50 / 1M output tokens
- Claude 3.5 Sonnet: $3.00 / 1M input tokens, $15.00 / 1M output tokens

**Hints**:
<details>
<summary>Hint 1</summary>
Cost = (input_tokens / 1,000,000) * input_price + (output_tokens / 1,000,000) * output_price
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def calculate_cost(
    input_tokens: int,
    output_tokens: int,
    input_price_per_million: float,
    output_price_per_million: float
) -> float:
    """
    Calculate cost of API call.
    
    Parameters
    ----------
    input_tokens : int
        Number of input tokens
    output_tokens : int
        Number of output tokens
    input_price_per_million : float
        Price per million input tokens
    output_price_per_million : float
        Price per million output tokens
    
    Returns
    -------
    float
        Total cost in dollars
    """
    # TODO: Implement this function
    pass

exercise_1_3_answer = calculate_cost  # Don't change this line

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_1_3():
    assert exercise_1_3_answer is not None, "Don't forget to implement your solution!"
    
    # Test case 1: 1000 input, 1000 output at $1/$2 per million
    cost = exercise_1_3_answer(1000, 1000, 1.0, 2.0)
    expected = (1000/1_000_000 * 1.0) + (1000/1_000_000 * 2.0)
    assert abs(cost - expected) < 0.0001, f"Expected {expected}, got {cost}"
    
    # Test case 2: Zero tokens
    cost = exercise_1_3_answer(0, 0, 3.0, 15.0)
    assert cost == 0.0, "Zero tokens should cost $0"
    
    # Test case 3: Claude 3.5 pricing (1M input, 100k output)
    cost = exercise_1_3_answer(1_000_000, 100_000, 3.0, 15.0)
    expected = 3.0 + (0.1 * 15.0)  # $3 + $1.50 = $4.50
    assert abs(cost - expected) < 0.01, f"Expected ~${expected}, got ${cost}"
    
    print("✓ Exercise 1.3 passed!")

test_exercise_1_3()

## Solutions

Below are reference implementations for the exercises. Try to solve them yourself before looking!

In [ ]:
# SOLUTION 1.1: Unified API Client

def unified_llm_call_solution(provider: str, prompt: str, api_key: str) -> Dict:
    """
    Call LLM API with unified interface.
    """
    if provider == 'openai':
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return {
            "content": response.choices[0].message.content,
            "tokens": response.usage.total_tokens,
            "model": response.model
        }
    
    elif provider == 'anthropic':
        import anthropic
        client = anthropic.Anthropic(api_key=api_key)
        response = client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}]
        )
        return {
            "content": response.content[0].text,
            "tokens": response.usage.input_tokens + response.usage.output_tokens,
            "model": response.model
        }
    
    else:
        raise ValueError(f"Unknown provider: {provider}")

# SOLUTION 1.2: Token Counter

def estimate_tokens_solution(text: str) -> int:
    """
    Estimate token count for text.
    """
    import math
    word_count = len(text.split())
    return math.ceil(word_count * 1.3)

# SOLUTION 1.3: Cost Calculator

def calculate_cost_solution(
    input_tokens: int,
    output_tokens: int,
    input_price_per_million: float,
    output_price_per_million: float
) -> float:
    """
    Calculate cost of API call.
    """
    input_cost = (input_tokens / 1_000_000) * input_price_per_million
    output_cost = (output_tokens / 1_000_000) * output_price_per_million
    return input_cost + output_cost

## Summary

**Key Takeaways**:

1. **API Setup**: Always use environment variables for API keys, never hardcode them
2. **Provider Differences**: OpenAI and Anthropic have similar but distinct API structures
3. **Error Handling**: Implement retry logic with exponential backoff for production use
4. **Token Awareness**: Understanding tokens is crucial for cost management and context limits
5. **Unified Interfaces**: Abstracting provider differences makes code more maintainable

**What's Next**:
- Module 0.2: Deep dive into tokenization mechanics
- Module 0.3: Advanced prompt engineering techniques
- Module 1: Building your first agents

**Additional Resources**:
- [OpenAI API Documentation](https://platform.openai.com/docs/api-reference)
- [Anthropic API Documentation](https://docs.anthropic.com/claude/reference/getting-started-with-the-api)
- [OpenAI Tokenizer Tool](https://platform.openai.com/tokenizer)

---

Congratulations on completing Module 0.1! You now have the foundation to interact with multiple LLM providers.